# Phase 2 : diagnostic de la qualité des données

Ce notebook regroupe les contrôles de la phase 2, avec une section par ticket Jira. Il mesure les problèmes sans rien modifier dans les données : les corrections se font en phase 3.

Avant de commiter : **Clear All Outputs**, car certaines sorties peuvent afficher des données sensibles.

## P2-17 : étiquettes de fraude

La table `fraud_labels.json`, ajoutée par l'enseignant, indique pour une partie des transactions s'il s'agit d'une fraude.

Critère de fin : couverture, taux de fraude et cohérence avec les transactions mesurés et documentés (voir `docs/P2-17_etiquettes_fraude.md`).

In [ ]:
from src.load_data import load_transactions, load_fraud_labels
from src.diagnostic import check_fraud_labels, add_fraud_label, fraud_summary_by

In [ ]:
# Seules les colonnes utiles au contrôle sont lues, pour économiser la mémoire
df_tx = load_transactions(usecols=["id", "date", "use_chip"])
df_labels = load_fraud_labels()

print(df_labels.shape)
df_labels.head()

In [ ]:
check_fraud_labels(df_labels, df_tx)

### Couverture et taux de fraude par année

In [ ]:
df_lab = add_fraud_label(df_tx, df_labels)
df_lab["annee"] = df_lab["date"].dt.year
fraud_summary_by(df_lab, "annee")

### Couverture et taux de fraude par canal de paiement

In [ ]:
fraud_summary_by(df_lab, "use_chip")

### Constats

- 8 914 963 transactions sont étiquetées, soit 67,0 % du total. Aucun identifiant n'est en double, et tous existent dans les transactions.
- 13 332 sont des fraudes, soit 0,150 % des transactions étiquetées : les deux classes sont très déséquilibrées.
- La part étiquetée est comprise entre 66,9 % et 67,1 % selon l'année, et vaut 67,0 % pour chaque canal. Les étiquettes semblent donc tirées au hasard : la partie étiquetée représente bien l'ensemble.
- Le taux de fraude varie beaucoup selon l'année : de 0,004 % en 2011 (37 fraudes) à 0,309 % en 2010. Il faudra en tenir compte pour séparer les données d'entraînement et de test (P7-08).
- Le taux de fraude est de 0,841 % en ligne, 0,099 % par puce et 0,029 % par bande magnétique. Le canal de paiement sera un indicateur à suivre en phase 4.